# Daily Challenge — Building an Agent with LangGraph and the Gemini API

**Week 9 — Bootcamp GenAI & Machine Learning 2026**

In this notebook we build **BaristaBot**, a stateful cafe ordering system using:
- **LangGraph** for the graph-based state machine
- **Gemini API** (via LangChain) as the language model
- **Tool calling** for menu lookup and order management

---
## Step 1 — Install Dependencies

In [ ]:
%pip install -qU "langgraph==1.0.5" "langchain-google-genai==4.1.2" "google-genai==1.56.0"

---
## Step 2 — Set Up the Gemini API Key

In [ ]:
import os

# In Google Colab: add GOOGLE_API_KEY via Secrets (key icon in sidebar)
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    # Fallback: set the key directly
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
print("API key configured ✅" if GOOGLE_API_KEY else "⚠️  GOOGLE_API_KEY not set")

---
## Step 3 — Define State and System Instructions

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages


class OrderState(TypedDict):
    """State representing the customer's order conversation."""
    # Conversation history — add_messages appends rather than replaces
    messages: Annotated[list, add_messages]
    # The customer's in-progress order (list of strings)
    order: list[str]
    # Flag set to True once the order is placed
    finished: bool


# System instruction: tone, rules, and tool-usage playbook
BARISTABOT_SYSINT = (
    "system",
    "You are a BaristaBot, an interactive cafe ordering system. A human will talk to you about the "
    "available products you have and you will answer any questions about menu items (and only about "
    "menu items - no off-topic discussion, but you can chat about the products and their history). "
    "The customer will place an order for 1 or more items from the menu, which you will structure "
    "and send to the ordering system after confirming the order with the human. "
    "\n\n"
    "Add items to the customer's order with add_to_order, and reset the order with clear_order. "
    "To see the contents of the order so far, call get_order (this is shown to you, not the user) "
    "Always confirm_order with the user (double-check) before calling place_order. Calling confirm_order will "
    "display the order items to the user and returns their response to seeing the list. Their response may contain modifications. "
    "Always verify and respond with drink and modifier names from the MENU before adding them to the order. "
    "If you are unsure a drink or modifier matches those on the MENU, ask a question to clarify or redirect. "
    "You only have the modifiers listed on the menu. "
    "Once the customer has finished ordering items, Call confirm_order to ensure it is correct then make "
    "any necessary updates and then call place_order. Once place_order has returned, thank the user and "
    "say goodbye!",
)

WELCOME_MSG = "Welcome to the BaristaBot cafe. Type `q` to quit. How may I serve you today?"

print("State and instructions defined ✅")

---
## Step 4 — Single-Turn Chatbot Graph

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-latest")


def chatbot(state: OrderState) -> OrderState:
    """The chatbot itself. A simple wrapper around the model's chat interface."""
    message_history = [BARISTABOT_SYSINT] + state["messages"]
    return {"messages": [llm.invoke(message_history)]}


# Build the single-node graph
graph_builder = StateGraph(OrderState)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")   # entry point → chatbot

chat_graph = graph_builder.compile()
print("Single-turn graph compiled ✅")

---
## Step 5 — Visualise the Graph

In [ ]:
from IPython.display import Image

Image(chat_graph.get_graph().draw_mermaid_png())

---
## Step 6 — Run the Single-Turn Graph

In [ ]:
from pprint import pprint

# First turn
user_msg = "Hello! What kind of drinks do you offer?"
state = chat_graph.invoke({"messages": [("user", user_msg)], "order": [], "finished": False})

print("=== Turn 1 ===")
for msg in state["messages"]:
    print(f"{type(msg).__name__}: {msg.content[:200]}")

---
## Step 7 — Second Conversational Turn

In [ ]:
# Second turn — append a follow-up message and invoke again
user_msg = "Do you have any tea options?"
state["messages"].append(("user", user_msg))
state = chat_graph.invoke(state)

print("=== Turn 2 ===")
for msg in state["messages"]:
    print(f"{type(msg).__name__}: {msg.content[:200]}")

---
## Step 8 — Add a Human Node (Interactive Loop)

Instead of calling `invoke` manually each turn, we let LangGraph loop between `chatbot` and `human` nodes.

In [ ]:
from langchain_core.messages.ai import AIMessage


def human_node(state: OrderState) -> OrderState:
    """Display the last model message and receive the user's input."""
    last_msg = state["messages"][-1]
    print("Model:", last_msg.content)

    user_input = input("User: ")

    # Flag conversation as finished if the user wants to quit
    if user_input in {"q", "quit", "exit", "goodbye"}:
        state["finished"] = True

    return state | {"messages": [("user", user_input)]}


def chatbot_with_welcome_msg(state: OrderState) -> OrderState:
    """Chatbot with welcome message on first turn."""
    if state["messages"]:
        new_output = llm.invoke([BARISTABOT_SYSINT] + state["messages"])
    else:
        new_output = AIMessage(content=WELCOME_MSG)
    return state | {"messages": [new_output]}


# Build the graph with chatbot ↔ human loop
graph_builder = StateGraph(OrderState)
graph_builder.add_node("chatbot", chatbot_with_welcome_msg)
graph_builder.add_node("human", human_node)

graph_builder.add_edge(START, "chatbot")      # start → chatbot
graph_builder.add_edge("chatbot", "human")    # chatbot always goes to human

print("Graph with human node defined ✅")

---
## Step 9 — Add Conditional Exit Edge

In [ ]:
from typing import Literal


def maybe_exit_human_node(state: OrderState) -> Literal["chatbot", "__end__"]:
    """Route to chatbot unless the user flagged the conversation as finished."""
    if state.get("finished", False):
        return END
    else:
        return "chatbot"


graph_builder.add_conditional_edges("human", maybe_exit_human_node)

chat_with_human_graph = graph_builder.compile()

Image(chat_with_human_graph.get_graph().draw_mermaid_png())

---
## Step 10 — Add a Live Menu Tool

The `get_menu` tool is **stateless** — it only reads data — so it can be called automatically via a `ToolNode`.

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode


@tool
def get_menu() -> str:
    """Provide the latest up-to-date menu."""
    return """
    MENU:
    Coffee Drinks:
    Espresso
    Americano
    Cold Brew

    Coffee Drinks with Milk:
    Latte
    Cappuccino
    Cortado
    Macchiato
    Mocha
    Flat White

    Tea Drinks:
    English Breakfast Tea
    Green Tea
    Earl Grey

    Tea Drinks with Milk:
    Chai Latte
    Matcha Latte
    London Fog

    Other Drinks:
    Steamer
    Hot Chocolate

    Modifiers:
    Milk options: Whole, 2%, Oat, Almond, 2% Lactose Free; Default: whole
    Espresso shots: Single, Double, Triple, Quadruple; Default: Double
    Caffeine: Decaf, Regular; Default: Regular
    Hot-Iced: Hot, Iced; Default: Hot
    Sweeteners: vanilla sweetener, hazelnut sweetener, caramel sauce, chocolate sauce, sugar free vanilla sweetener
    Special requests: extra hot, one pump, half caff, extra foam, etc.

    'dirty' means add a shot of espresso to a drink that doesn't usually have it.
    'Regular milk' = whole milk.
    'Sweetened' means add regular sugar, not a sweetener.

    NOTE: Soy milk is out of stock today.
    """


# Wrap the tool and bind it to the LLM
tools = [get_menu]
tool_node = ToolNode(tools)
llm_with_tools = llm.bind_tools(tools)


def maybe_route_to_tools(state: OrderState) -> Literal["tools", "human"]:
    """Route to tools node if the model made a tool call, otherwise to human."""
    if not (msgs := state.get("messages", [])):
        raise ValueError(f"No messages found when parsing state: {state}")
    msg = msgs[-1]
    if hasattr(msg, "tool_calls") and len(msg.tool_calls) > 0:
        return "tools"
    else:
        return "human"


def chatbot_with_tools(state: OrderState) -> OrderState:
    """Chatbot that knows about tools."""
    defaults = {"order": [], "finished": False}
    if state["messages"]:
        new_output = llm_with_tools.invoke([BARISTABOT_SYSINT] + state["messages"])
    else:
        new_output = AIMessage(content=WELCOME_MSG)
    return defaults | state | {"messages": [new_output]}


# Build the graph with the menu tool
graph_builder = StateGraph(OrderState)
graph_builder.add_node("chatbot", chatbot_with_tools)
graph_builder.add_node("human", human_node)
graph_builder.add_node("tools", tool_node)

graph_builder.add_conditional_edges("chatbot", maybe_route_to_tools)  # chatbot → tools or human
graph_builder.add_conditional_edges("human", maybe_exit_human_node)   # human → chatbot or END
graph_builder.add_edge("tools", "chatbot")                             # tools always → chatbot
graph_builder.add_edge(START, "chatbot")

graph_with_menu = graph_builder.compile()
Image(graph_with_menu.get_graph().draw_mermaid_png())

---
## Step 11 — Add Ordering Tools and Order Node

These tools are **stateful** — they modify the order — so they are handled by a dedicated `order_node` that can directly edit the graph state.

In [ ]:
from collections.abc import Iterable
from random import randint
from langchain_core.messages.tool import ToolMessage


# Ordering tool schemas — bodies are intentionally empty;
# the actual logic lives in order_node below.
@tool
def add_to_order(drink: str, modifiers: Iterable[str]) -> str:
    """Adds the specified drink to the customer's order, including any modifiers.

    Returns:
      The updated order in progress.
    """


@tool
def confirm_order() -> str:
    """Asks the customer if the order is correct.

    Returns:
      The user's free-text response.
    """


@tool
def get_order() -> str:
    """Returns the user's order so far. One item per line."""


@tool
def clear_order():
    """Removes all items from the user's order."""


@tool
def place_order() -> int:
    """Sends the order to the barista for fulfillment.

    Returns:
      The estimated number of minutes until the order is ready.
    """


def order_node(state: OrderState) -> OrderState:
    """The ordering node — directly edits order state based on tool calls."""
    tool_msg = state["messages"][-1]    # last message contains the tool calls
    order = state.get("order", [])       # current in-progress order
    outbound_msgs = []
    order_placed = False

    for tool_call in tool_msg.tool_calls:

        if tool_call["name"] == "add_to_order":
            modifiers = tool_call["args"].get("modifiers", [])
            modifier_str = ", ".join(modifiers) if modifiers else "no modifiers"
            order.append(f'{tool_call["args"]["drink"]} ({modifier_str})')
            response = "\n".join(order)

        elif tool_call["name"] == "confirm_order":
            # Show the exact order to the user before they confirm
            print("Your order:")
            if not order:
                print("  (no items)")
            for drink in order:
                print(f"  {drink}")
            response = input("Is this correct? ")

        elif tool_call["name"] == "get_order":
            response = "\n".join(order) if order else "(no order)"

        elif tool_call["name"] == "clear_order":
            order.clear()
            response = "Order cleared."

        elif tool_call["name"] == "place_order":
            order_text = "\n".join(order)
            print("Sending order to kitchen!")
            print(order_text)
            order_placed = True
            response = randint(1, 5)  # ETA in minutes

        else:
            raise NotImplementedError(f'Unknown tool call: {tool_call["name"]}')

        outbound_msgs.append(
            ToolMessage(
                content=str(response),
                name=tool_call["name"],
                tool_call_id=tool_call["id"],
            )
        )

    return {"messages": outbound_msgs, "order": order, "finished": order_placed}


print("Ordering tools and order_node defined ✅")

---
## Step 12 — Build the Complete BaristaBot Graph

In [ ]:
# Auto-tools: handled automatically by ToolNode
auto_tools = [get_menu]
tool_node = ToolNode(auto_tools)

# Order-tools: handled by the custom order_node
order_tools = [add_to_order, confirm_order, get_order, clear_order, place_order]

# LLM must know about ALL tools so it can call them
llm_with_tools = llm.bind_tools(auto_tools + order_tools)


def chatbot_with_tools(state: OrderState) -> OrderState:
    """Final chatbot node — uses all tools."""
    defaults = {"order": [], "finished": False}
    if state["messages"]:
        new_output = llm_with_tools.invoke([BARISTABOT_SYSINT] + state["messages"])
    else:
        new_output = AIMessage(content=WELCOME_MSG)
    return defaults | state | {"messages": [new_output]}


def maybe_route_to_tools(state: OrderState) -> str:
    """Route chatbot output to: auto tools, ordering tools, human, or END."""
    if not (msgs := state.get("messages", [])):
        raise ValueError(f"No messages found when parsing state: {state}")

    msg = msgs[-1]

    # If the order is placed, exit gracefully
    if state.get("finished", False):
        return END

    # If there are tool calls, decide which node handles them
    if hasattr(msg, "tool_calls") and len(msg.tool_calls) > 0:
        auto_tool_names = set(tool_node.tools_by_name.keys())
        if any(tc["name"] in auto_tool_names for tc in msg.tool_calls):
            return "tools"     # stateless auto-tool (e.g. get_menu)
        else:
            return "ordering"  # stateful order tool

    return "human"  # no tool calls → hand off to human


# Build the complete graph
graph_builder = StateGraph(OrderState)

# Nodes
graph_builder.add_node("chatbot", chatbot_with_tools)
graph_builder.add_node("human", human_node)
graph_builder.add_node("tools", tool_node)
graph_builder.add_node("ordering", order_node)

# Edges
graph_builder.add_conditional_edges("chatbot", maybe_route_to_tools)  # chatbot → {tools, ordering, human, END}
graph_builder.add_conditional_edges("human", maybe_exit_human_node)   # human → {chatbot, END}
graph_builder.add_edge("tools", "chatbot")                             # auto-tools → chatbot
graph_builder.add_edge("ordering", "chatbot")                          # order-tools → chatbot
graph_builder.add_edge(START, "chatbot")                               # START → chatbot

graph_with_order_tools = graph_builder.compile()

Image(graph_with_order_tools.get_graph().draw_mermaid_png())

---
## Run the Complete BaristaBot

> **Note:** This cell is interactive — it uses `input()` to receive user messages.  
> Type `q` to quit, or place a complete order to exit naturally.
>
> **Things to try:**
> - Order a drink (e.g. "I'd like an oat milk latte")
> - Ask about the menu ("What teas do you have?")
> - Modify your order before confirming
> - Ask an off-topic question and see the bot stay on-topic

In [ ]:
from pprint import pprint

# Higher recursion limit for complex multi-step orders
config = {"recursion_limit": 100}

state = graph_with_order_tools.invoke(
    {"messages": [], "order": [], "finished": False},
    config
)

print("\n=== Final state ===")
print("Order:", state.get("order"))
print("Finished:", state.get("finished"))

---
## Architecture Summary

```
START
  └──► chatbot  ──► [tool call? auto-tool?]──► tools ──┐
           │                                            │
           │        [tool call? order-tool?]──► ordering┘
           │                                    (both return to chatbot)
           │
           └──► [no tool call] ──► human
                                     │
                                     ├── [finished=True] ──► END
                                     └── [finished=False] ──► chatbot
```

| Node | Role |
|------|------|
| `chatbot` | Calls Gemini API with system prompt + conversation history |
| `human` | Displays AI response, collects user input |
| `tools` | Auto-executes stateless tools (`get_menu`) via `ToolNode` |
| `ordering` | Executes stateful order tools (`add_to_order`, `confirm_order`, `place_order`, …) |

### Key LangGraph Concepts Used

- **`TypedDict` state schema** — typed, shared state object passed between every node
- **`add_messages` annotation** — appends messages to history instead of replacing them
- **`@tool` decorator** — declares tool schemas for the LLM without implementing the body
- **`ToolNode`** — automatically dispatches and handles stateless tool calls
- **Conditional edges** — `maybe_route_to_tools` and `maybe_exit_human_node` implement branching and loop exit
- **`recursion_limit`** — prevents infinite loops during complex multi-step conversations